## 01 Data Preparation

This notebook prepares the raw JAMK course dataset for curriculum-overlap detection.

The notebook has two goals:

1. **Thesis goal:** create a transparent, defensible course representation for embedding-based overlap detection.
2. **PoC goal:** produce a clean and reusable dataset that later notebooks and the final UI can use without ad hoc fixes.

This notebook is treated as the **source of truth** for downstream work.

Main tasks:
- load and validate the raw JSON dataset
- audit dataset structure and quality
- flatten nested fields into explicit columns
- normalize text conservatively
- create embedding-ready text variants
- prepare fair review text for manual labeling
- generate diagnostics that justify preprocessing choices
- save the cleaned dataset for later notebooks

Main output:
- `data/courses_clean.csv`

### Final representation strategy

This notebook creates two embedding variants:

- **`embedding_text_main`**  
  a conservative core representation of course meaning

- **`embedding_text_enriched`**  
  a broader representation that preserves additional curriculum signals when they exist

This design reflects a deliberate methodological choice:

- the **main** variant should remain stable and comparable across the full dataset
- the **enriched** variant should preserve sparse but potentially meaningful signals such as learning outcomes, teamwork-related competences, multidisciplinary competences, and prerequisite relations

The purpose is not to assume that broader text is always better.  
The purpose is to test whether these additional curriculum signals improve overlap detection while keeping the core benchmark representation clean and interpretable.

In [118]:
# Cell 01 - Import libraries and define project paths

import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

RAW_JSON_PATH = DATA_DIR / "jamk_courses.json"
CLEAN_CSV_PATH = DATA_DIR / "courses_clean.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())
print("Raw JSON path:", RAW_JSON_PATH.resolve())
print("Clean CSV path:", CLEAN_CSV_PATH.resolve())
print("Raw JSON exists:", RAW_JSON_PATH.exists())

Project root: C:\Users\user\Downloads\curriculum-overlap-poc
Data directory: C:\Users\user\Downloads\curriculum-overlap-poc\data
Raw JSON path: C:\Users\user\Downloads\curriculum-overlap-poc\data\jamk_courses.json
Clean CSV path: C:\Users\user\Downloads\curriculum-overlap-poc\data\courses_clean.csv
Raw JSON exists: True


### 1. Load and validate the raw dataset

This section loads the raw course JSON and checks that the file has the expected top-level structure:
- a list of courses
- each course represented as a dictionary

The purpose is not only to load data, but also to confirm that the raw input is structurally suitable for later processing.

In [119]:
# Cell 02 - Load raw JSON safely

if not RAW_JSON_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_JSON_PATH.resolve()}")

with open(RAW_JSON_PATH, "r", encoding="utf-8") as f:
    courses = json.load(f)

print("Dataset type:", type(courses))
print("Number of courses:", len(courses))

if not isinstance(courses, list):
    raise TypeError("Expected the raw dataset to be a list of course dictionaries.")

if len(courses) == 0:
    raise ValueError("The raw dataset is empty.")

print("First item type:", type(courses[0]))

if not all(isinstance(item, dict) for item in courses):
    raise TypeError("All items in the raw dataset should be dictionaries.")

Dataset type: <class 'list'>
Number of courses: 50
First item type: <class 'dict'>


In [120]:
# Cell 03 - Preview one raw course example

courses[0]

{'degree_programme': 'Information and Communication Technology',
 'course_name': 'Data Networks',
 'course_unit_code': 'TT00CD70',
 'credits': 5,
 'teaching_language': ['Finnish', 'English'],
 'responsible_person': ['Pasi Hyytiäinen'],
 'objective': ['You learn the structure and protocols of data networks used by computers and other end devices.',
  'You know the principles of planning a data network.',
  'You are familiar with protocols and data transmission principles used on the Internet.',
  'You know the layers of the OSI model and identify the technologies and protocols used on each layer.',
  'You can design a local area network using switches, routers, workstations, and servers.',
  'You are able to design cabling, switching topology, and routing scheme to run the data network.',
  'You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations.',
  'You can change, handle, and save the configuration used in network devices.',

In [121]:
# Cell 04 - Convert raw JSON list into a dataframe

raw_df = pd.DataFrame(courses)

print("Raw dataframe shape:", raw_df.shape)
raw_df.head(3)

Raw dataframe shape: (50, 12)


,degree_programme,course_name,course_unit_code,credits,teaching_language,responsible_person,objective,competences,learning_outcomes,content,prerequisites,assessment
0,Information and Communication Technology,Data Networks,TT00CD70,5,"[Finnish, English]",[Pasi Hyytiäinen],"[You learn the structure and protocols of data networks used by computers and other end devices., You know the principles of planning a data network., You a...","{'knowledge_and_understanding': ['You know the main modeling methods of data networks.', 'You have knowledge and understanding of data networks and their ef...",[],"[In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication., You will mast...",[Linux Basics],"{'grading_scale': '1-5', 'criteria': {'1': 'The student knows the theory on network protocols sufficiently and is able to recognize some sections of existin..."
1,Information and Communication Technology,Linux Basics,TT00CD71,4,"[Finnish, English]","[Juho Pekki, Teemu Siikaniemi]","[After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage ...",{'knowledge_and_understanding': ['You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing ...,[],"[In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be a...",[],"{'grading_scale': '1-5', 'criteria': {'1': 'You are familiar with the basic concepts of the Linux operating system and can use basic commands. You know how ..."
2,Information and Communication Technology,Servers and containers,TT00CD72,5,"[Finnish, English]","[Juho Pekki, Teemu Siikaniemi]",[You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able...,{'knowledge_and_understanding': ['You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service m...,[],"[In this course, you will learn the basic concepts of servers and cloud computing, and differentiate between different cloud service models. You will master...","[Linux Basics, Data Networks, Databases]","{'grading_scale': '1-5', 'criteria': {'1': 'You are to some extent familiar with the basic concepts of the servers and virtualization and know how to create..."


In [122]:
# Cell 05 - Show raw columns

print("Raw columns:")
for col in raw_df.columns:
    print("-", col)

Raw columns:
- degree_programme
- course_name
- course_unit_code
- credits
- teaching_language
- responsible_person
- objective
- competences
- learning_outcomes
- content
- prerequisites
- assessment


### 2. Initial structural audit

Before building any embedding text, it is important to inspect the raw dataset for:
- missing fields
- empty-but-not-null values
- duplicate identifiers
- unexpected datatypes

This matters for both thesis transparency and PoC reliability.

In [123]:
# Cell 06 - Check true missing values recognized by pandas

true_missing_summary = raw_df.isna().sum().sort_values(ascending=False)
true_missing_summary

degree_programme      0
course_name           0
course_unit_code      0
credits               0
teaching_language     0
responsible_person    0
objective             0
competences           0
learning_outcomes     0
content               0
prerequisites         0
assessment            0
dtype: int64

In [124]:
# Cell 07 - Define helper functions for structural checks

def is_empty_list(value):
    return isinstance(value, list) and len(value) == 0

def is_empty_string(value):
    return isinstance(value, str) and value.strip() == ""

def safe_get_dict_value(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

def get_list_from_dict(d, key):
    value = safe_get_dict_value(d, key, [])
    return value if isinstance(value, list) else []

def get_string_from_dict(d, key):
    value = safe_get_dict_value(d, key, "")
    return value if isinstance(value, str) else ""

In [125]:
# Cell 08 - Check empty-but-not-null values inside nested structures

empty_checks = {
    "empty_prerequisites": raw_df["prerequisites"].apply(is_empty_list).sum(),
    "empty_learning_outcomes": raw_df["learning_outcomes"].apply(is_empty_list).sum(),
    "empty_objective": raw_df["objective"].apply(is_empty_list).sum(),
    "empty_content": raw_df["content"].apply(is_empty_list).sum(),
    "empty_responsible_person": raw_df["responsible_person"].apply(is_empty_list).sum(),
    "empty_teaching_language": raw_df["teaching_language"].apply(is_empty_list).sum(),
    "empty_competences_knowledge_and_understanding": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "knowledge_and_understanding"))
    ).sum(),
    "empty_competences_engineering_practice": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "engineering_practice"))
    ).sum(),
    "empty_competences_applying_technology_to_practice": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "applying_technology_to_practice"))
    ).sum(),
    "empty_competences_multidisciplinary_competence": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "multidisciplinary_competence"))
    ).sum(),
    "empty_competences_communication_and_teamwork": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "communication_and_teamwork"))
    ).sum(),
    "empty_competences_investigations_and_information_retrieval": raw_df["competences"].apply(
        lambda x: is_empty_list(get_list_from_dict(x, "investigations_and_information_retrieval"))
    ).sum(),
    "empty_assessment_grading_scale": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_string_from_dict(x, "grading_scale"))
    ).sum(),
    "empty_assessment_pass_fail_criteria": raw_df["assessment"].apply(
        lambda x: is_empty_string(get_string_from_dict(x, "pass_fail_criteria"))
    ).sum(),
}

pd.Series(empty_checks).sort_values(ascending=False)

empty_competences_investigations_and_information_retrieval    49
empty_competences_applying_technology_to_practice             49
empty_learning_outcomes                                       47
empty_assessment_pass_fail_criteria                           47
empty_competences_multidisciplinary_competence                46
empty_competences_communication_and_teamwork                  45
empty_prerequisites                                            5
empty_competences_engineering_practice                         5
empty_competences_knowledge_and_understanding                  3
empty_objective                                                0
empty_teaching_language                                        0
empty_responsible_person                                       0
empty_content                                                  0
empty_assessment_grading_scale                                 0
dtype: int64

In [127]:
# Cell 09 - Check duplicate course codes

duplicate_codes = raw_df[raw_df["course_unit_code"].duplicated(keep=False)].sort_values("course_unit_code")
duplicate_codes[["course_unit_code", "course_name"]]

,course_unit_code,course_name


In [128]:
# Cell 10 - Check duplicate course names

duplicate_names = raw_df[raw_df["course_name"].duplicated(keep=False)].sort_values("course_name")
duplicate_names[["course_unit_code", "course_name"]]

,course_unit_code,course_name


In [129]:
# Cell 11 - Check datatype consistency of important raw fields

datatype_audit = pd.DataFrame({
    "column": raw_df.columns,
    "python_types_seen": [
        sorted({type(v).__name__ for v in raw_df[col]})
        for col in raw_df.columns
    ]
})

datatype_audit

,column,python_types_seen
0,degree_programme,[str]
1,course_name,[str]
2,course_unit_code,[str]
3,credits,[int]
4,teaching_language,[list]
5,responsible_person,[list]
6,objective,[list]
7,competences,[dict]
8,learning_outcomes,[list]
9,content,[list]


### 3. Flatten nested fields into explicit columns

The raw JSON contains nested lists and dictionaries.  
This section converts those structures into explicit columns that are easier to inspect, clean, reuse, and save.

This also makes the preprocessing steps auditable: later notebooks can trace exactly which textual signals were included in the final representation.

In [131]:
# Cell 12 - Copy raw dataframe and flatten nested fields

df = raw_df.copy()

df["competences_knowledge_and_understanding"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "knowledge_and_understanding")
)

df["competences_engineering_practice"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "engineering_practice")
)

df["competences_applying_technology_to_practice"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "applying_technology_to_practice")
)

df["competences_multidisciplinary_competence"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "multidisciplinary_competence")
)

df["competences_communication_and_teamwork"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "communication_and_teamwork")
)

df["competences_investigations_and_information_retrieval"] = df["competences"].apply(
    lambda x: get_list_from_dict(x, "investigations_and_information_retrieval")
)

df["assessment_grading_scale"] = df["assessment"].apply(
    lambda x: get_string_from_dict(x, "grading_scale")
)

df["assessment_pass_fail_criteria"] = df["assessment"].apply(
    lambda x: get_string_from_dict(x, "pass_fail_criteria")
)

df[[
    "course_name",
    "competences_knowledge_and_understanding",
    "competences_engineering_practice",
    "competences_applying_technology_to_practice",
    "competences_multidisciplinary_competence",
    "competences_communication_and_teamwork",
    "competences_investigations_and_information_retrieval",
    "learning_outcomes",
    "assessment_grading_scale",
    "assessment_pass_fail_criteria"
]].head(3)

,course_name,competences_knowledge_and_understanding,competences_engineering_practice,competences_applying_technology_to_practice,competences_multidisciplinary_competence,competences_communication_and_teamwork,competences_investigations_and_information_retrieval,learning_outcomes,assessment_grading_scale,assessment_pass_fail_criteria
0,Data Networks,"[You know the main modeling methods of data networks., You have knowledge and understanding of data networks and their effects., You know the basic principl...",[You can apply data network skills to communication between devices.],[],[],[],[],[],1-5,The student has sufficient knowledge of the theory on network protocols and is able to recognize some sections of existing solutions. The student can design...
1,Linux Basics,[You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.],[You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions...,[],[],[],[],[],1-5,
2,Servers and containers,[You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pr...,"[You are able to utilize basic cloud services and create virtual machines. You are able to build, run, and manage containers, and use orchestration tools to...",[],[],[],[],[],1-5,


### 4. Conservative text cleaning helpers

This notebook uses **conservative preprocessing**.

Included:
- whitespace normalization
- simple bullet cleanup
- safe joining of list fields into readable text
- light boundary preservation between list items when needed

Avoided on purpose:
- stop-word removal
- stemming or lemmatization
- aggressive phrase stripping
- lowercasing of final embedding text
- injection of full prerequisite course descriptions

The goal is to preserve course meaning as faithfully as possible while avoiding unnecessary preprocessing choices that would be difficult to justify in a thesis setting.

In [132]:
# Cell 13 - Define text cleaning helper functions

def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def remove_leading_bullets(text: str) -> str:
    text = re.sub(r"^[\-\*\u2022]+\s*", "", text)
    return text.strip()

def clean_text_value(value) -> str:
    """
    Clean a single value into a compact readable string.
    Keeps natural language intact for embedding and review use.
    """
    if value is None:
        return ""

    if not isinstance(value, str):
        value = str(value)

    text = value.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = remove_leading_bullets(text)
    text = normalize_whitespace(text)

    return text

def ensure_sentence_like_boundary(text: str) -> str:
    """
    Add a soft boundary between list items when the item does not already end
    with common sentence punctuation. This preserves readability without
    aggressively rewriting the source text.
    """
    if not text:
        return text

    if re.search(r"[.!?:;]$", text):
        return text

    return text + "."

def join_list_field(value, separator=" "):
    """
    Join list items into one clean string.
    Empty items are removed.
    For sentence-like separators, add a soft boundary so list items do not
    collapse into each other.
    """
    if isinstance(value, list):
        cleaned_items = [clean_text_value(item) for item in value]
        cleaned_items = [item for item in cleaned_items if item]

        if separator == " ":
            cleaned_items = [ensure_sentence_like_boundary(item) for item in cleaned_items]

        return separator.join(cleaned_items).strip()

    if isinstance(value, str):
        return clean_text_value(value)

    return ""

def normalize_for_matching(text: str) -> str:
    """
    Lowercased helper text for matching / lookup operations.
    Not intended for final embedding text.
    """
    text = clean_text_value(text).lower()
    text = re.sub(r"[^\w\s\-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def char_len(text) -> int:
    return len(clean_text_value(text))

def word_len(text) -> int:
    cleaned = clean_text_value(text)
    return len(cleaned.split()) if cleaned else 0

### 5. Create cleaned flat text columns

These columns serve two purposes:
- they make the course records readable and inspectable
- they provide the building blocks for embedding text and review text

The key design principle is to preserve **semantically rich curriculum content** while keeping the processing pipeline transparent and reproducible.

In [133]:
# Cell 14 - Create cleaned flat text columns

df["degree_programme_text"] = df["degree_programme"].apply(clean_text_value)
df["course_name_text"] = df["course_name"].apply(clean_text_value)
df["course_unit_code_text"] = df["course_unit_code"].apply(clean_text_value)

df["teaching_language_text"] = df["teaching_language"].apply(lambda x: join_list_field(x, separator=", "))
df["responsible_person_text"] = df["responsible_person"].apply(lambda x: join_list_field(x, separator=", "))

df["objective_text"] = df["objective"].apply(lambda x: join_list_field(x, separator=" "))

df["competences_knowledge_text"] = df["competences_knowledge_and_understanding"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_practice_text"] = df["competences_engineering_practice"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_applying_technology_text"] = df["competences_applying_technology_to_practice"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_multidisciplinary_text"] = df["competences_multidisciplinary_competence"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_communication_text"] = df["competences_communication_and_teamwork"].apply(
    lambda x: join_list_field(x, separator=" ")
)
df["competences_investigations_text"] = df["competences_investigations_and_information_retrieval"].apply(
    lambda x: join_list_field(x, separator=" ")
)

df["learning_outcomes_text"] = df["learning_outcomes"].apply(lambda x: join_list_field(x, separator=" "))
df["content_text"] = df["content"].apply(lambda x: join_list_field(x, separator=" "))
df["prerequisites_text"] = df["prerequisites"].apply(lambda x: join_list_field(x, separator=", "))

df[[
    "course_name",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text"
]].head(5)

,course_name,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text
0,Data Networks,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics
1,Linux Basics,"After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",
2,Servers and containers,You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able ...,You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pri...,"You are able to utilize basic cloud services and create virtual machines. You are able to build, run, and manage containers, and use orchestration tools to ...",,,,,,"In this course, you will learn the basic concepts of servers and cloud computing, and differentiate between different cloud service models. You will master ...","Linux Basics, Data Networks, Databases"
3,Digital Technology,You understand the basics of digital number systems and know how to apply the knowledge in practice. You can rationalize simple technical problems logically...,You know the concepts and principles related to digital technology. You can analyze logical structures.,You know how to plan and implement logical systems related to digital technology.,,,,,,In this course you will learn the basics of digital numbering systems and their practical application. You will be able to solve technical problems logicall...,
4,Information Systems and Architecture,"After completing the course, you understand what the concepts software, software service, service infrastructure and different integrations mean and how dif...",You can explain and identify the main components needed to implement information systems. You know the basic principles of system integrations. You understa...,You know how to describe and document a simple information system architecture using suitable methods and tools.,,,,,,"In this course you will learn about the concepts of software, software service, service infrastructure and integrations. You will understand how different c...","Data Networks, Basics of Programming, Databases"


In [134]:
# Cell 15 - Create helper normalized columns for matching and lightweight rule-based use

df["course_name_normalized"] = df["course_name"].apply(normalize_for_matching)
df["prerequisites_text_normalized"] = df["prerequisites_text"].apply(normalize_for_matching)

df[[
    "course_name",
    "course_name_normalized",
    "prerequisites_text",
    "prerequisites_text_normalized"
]].head(5)

,course_name,course_name_normalized,prerequisites_text,prerequisites_text_normalized
0,Data Networks,data networks,Linux Basics,linux basics
1,Linux Basics,linux basics,,
2,Servers and containers,servers and containers,"Linux Basics, Data Networks, Databases",linux basics data networks databases
3,Digital Technology,digital technology,,
4,Information Systems and Architecture,information systems and architecture,"Data Networks, Basics of Programming, Databases",data networks basics of programming databases


In [135]:
# Cell 16 - Define lightweight semantic cue helpers for enriched embedding text

def prefix_if_text_exists(prefix: str, text: str) -> str:
    cleaned = clean_text_value(text)
    if not cleaned:
        return ""
    return f"{prefix}{cleaned}"

def build_prerequisites_phrase(text: str) -> str:
    return prefix_if_text_exists("Prerequisites include: ", text)

def build_learning_outcomes_phrase(text: str) -> str:
    return prefix_if_text_exists("Learning outcomes include: ", text)

def build_applying_technology_phrase(text: str) -> str:
    return prefix_if_text_exists("Applying technology to practice includes: ", text)

def build_multidisciplinary_phrase(text: str) -> str:
    return prefix_if_text_exists("Multidisciplinary competence includes: ", text)

def build_communication_phrase(text: str) -> str:
    return prefix_if_text_exists("Communication and teamwork includes: ", text)

def build_investigations_phrase(text: str) -> str:
    return prefix_if_text_exists("Investigations and information retrieval includes: ", text)

### 6. Build text variants on purpose

A single text representation is rarely enough for a defensible benchmark.

This notebook creates two embedding variants:

- **embedding_text_main**  
  Primary benchmark representation. This is the default semantic course representation used for overlap detection.

- **embedding_text_enriched**  
  Extended variant that tests whether sparse competence fields, learning outcomes, and prerequisite information add useful curricular signal.

#### Embedding text design

The embedding variants are constructed from selected cleaned text fields.

##### `embedding_text_main`
This column contains the core semantic course representation:
- `course_name`
- `objective`
- `competences.knowledge_and_understanding`
- `competences.engineering_practice`
- `content`

##### `embedding_text_enriched`
This column extends the main representation with additional curriculum signals:
- all fields from `embedding_text_main`
- `competences.applying_technology_to_practice`
- `competences.multidisciplinary_competence`
- `competences.communication_and_teamwork`
- `competences.investigations_and_information_retrieval`
- `learning_outcomes`
- `prerequisites`

This split is intentional:

- **embedding_text_main** represents what the course is.
- **embedding_text_enriched** represents what the course is plus surrounding curriculum signals.

#### Reducing structural bias

The embedding variants are intentionally built **without heavy section labels** such as
`Course name:`, `Objective:`, or `Content:`.

This design reduces repeated structural boilerplate shared by all courses.

However, a few fields receive **light semantic cues** where the relationship matters for interpretation:
- prerequisites
- learning outcomes
- sparse competence categories included only in the enriched variant

By contrast, the human review text remains labeled, because labels improve readability during manual inspection and labeling.

In [136]:
# Cell 17 - Define text builders for embedding and review use

def build_unlabeled_text(parts):
    cleaned_parts = [clean_text_value(part) for part in parts]
    cleaned_parts = [part for part in cleaned_parts if part]
    return "\n".join(cleaned_parts)

def build_labeled_text(sections):
    parts = []
    for label, value in sections:
        cleaned_value = clean_text_value(value)
        if cleaned_value:
            parts.append(f"{label}: {cleaned_value}")
    return "\n".join(parts)

def build_embedding_text_main(row):
    return build_unlabeled_text([
        row["course_name_text"],
        row["objective_text"],
        row["competences_knowledge_text"],
        row["competences_practice_text"],
        row["content_text"],
    ])

def build_embedding_text_enriched(row):
    return build_unlabeled_text([
        row["course_name_text"],
        row["objective_text"],
        row["competences_knowledge_text"],
        row["competences_practice_text"],
        row["content_text"],
        build_applying_technology_phrase(row["competences_applying_technology_text"]),
        build_multidisciplinary_phrase(row["competences_multidisciplinary_text"]),
        build_communication_phrase(row["competences_communication_text"]),
        build_investigations_phrase(row["competences_investigations_text"]),
        build_learning_outcomes_phrase(row["learning_outcomes_text"]),
        build_prerequisites_phrase(row["prerequisites_text"]),
    ])

def build_review_text_compact(row):
    sections = [
        ("Course code", row["course_unit_code_text"]),
        ("Course name", row["course_name_text"]),
        ("Objective", row["objective_text"]),
        ("Content", row["content_text"]),
        ("Prerequisites", row["prerequisites_text"]),
    ]
    return build_labeled_text(sections)

def build_review_text_full(row):
    sections = [
        ("Course code", row["course_unit_code_text"]),
        ("Course name", row["course_name_text"]),
        ("Objective", row["objective_text"]),
        ("Knowledge and understanding", row["competences_knowledge_text"]),
        ("Engineering practice", row["competences_practice_text"]),
        ("Applying technology to practice", row["competences_applying_technology_text"]),
        ("Multidisciplinary competence", row["competences_multidisciplinary_text"]),
        ("Communication and teamwork", row["competences_communication_text"]),
        ("Investigations and information retrieval", row["competences_investigations_text"]),
        ("Learning outcomes", row["learning_outcomes_text"]),
        ("Content", row["content_text"]),
        ("Prerequisites", row["prerequisites_text"]),
    ]
    return build_labeled_text(sections)

In [137]:
# Cell 18 - Create embedding and review text variants

df["embedding_text_main"] = df.apply(build_embedding_text_main, axis=1)
df["embedding_text_enriched"] = df.apply(build_embedding_text_enriched, axis=1)

df["review_text_compact"] = df.apply(build_review_text_compact, axis=1)
df["review_text_full"] = df.apply(build_review_text_full, axis=1)

print(df.loc[0, "embedding_text_main"])

Data Networks
You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles of network design and how to 

In [138]:
# Cell 19 - Preview embedding and review variants for one course

preview_columns = [
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_full",
]

for col in preview_columns:
    print("=" * 100)
    print(col)
    print("-" * 100)
    print(df.loc[0, col])
    print()

embedding_text_main
----------------------------------------------------------------------------------------------------
Data Networks
You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
You know the main modeling methods of data networks. You have kn

### 7. Prepare fair review text for manual labeling

Manual labeling should not be based on a weaker representation than the one given to the models.

For that reason, this notebook prepares explicit review text fields that can later be shown in Notebook 02.

These fields are intended for human inspection, not for direct embedding:
- they keep labels for clarity
- they show the course structure more explicitly
- they help reduce annotation mistakes during pair review

In [139]:
# Cell 20 - Preview one human-readable full review text

print(df.loc[0, "review_text_full"])

Course code: TT00CD70
Course name: Data Networks
Objective: You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
Knowledge and understanding: You know the main modeling methods of data networks. You have knowledge and understanding of data networks and

### 8. Field-level audit

A strong preprocessing notebook should not only create text, but also explain **where the text comes from** and **how representation choices may affect later results**.

This section audits:
- coverage of important fields
- character and word contribution by field
- whether enrichment meaningfully expands the representation
- whether some fields are repetitive or sparse
- whether the enriched representation is activated by only a few special cases or by broader dataset patterns

This section is especially useful later when justifying preprocessing decisions in the thesis.

A key point in this notebook is that some curriculum signals are **structurally sparse but semantically important**.  
For that reason, both coverage and qualitative inspection are needed: low coverage alone is not sufficient reason to discard a field if it meaningfully represents a subset of courses.

In [140]:
# Cell 21 - Coverage audit of cleaned core fields

coverage_rows = []

for col in [
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_grading_scale",
    "assessment_pass_fail_criteria",
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_full",
]:
    non_empty_count = (df[col].fillna("").str.strip() != "").sum()
    coverage_rows.append({
        "column": col,
        "non_empty_rows": int(non_empty_count),
        "empty_rows": int(len(df) - non_empty_count),
        "coverage_ratio": round(non_empty_count / len(df), 4),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values(
    by=["coverage_ratio", "column"],
    ascending=[False, True]
).reset_index(drop=True)

coverage_df

,column,non_empty_rows,empty_rows,coverage_ratio
0,assessment_grading_scale,50,0,1.00
1,content_text,50,0,1.00
2,embedding_text_enriched,50,0,1.00
3,embedding_text_main,50,0,1.00
4,objective_text,50,0,1.00
5,review_text_full,50,0,1.00
6,competences_knowledge_text,47,3,0.94
7,competences_practice_text,45,5,0.90
8,prerequisites_text,45,5,0.90
9,competences_communication_text,5,45,0.10


In [141]:
# Cell 22 - Field length audit by course

length_audit_df = df[[
    "course_unit_code",
    "course_name",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_full",
]].copy()

text_cols_for_length = [
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_full",
]

for col in text_cols_for_length:
    length_audit_df[f"{col}_char_len"] = length_audit_df[col].apply(char_len)
    length_audit_df[f"{col}_word_len"] = length_audit_df[col].apply(word_len)

length_summary = length_audit_df[
    [c for c in length_audit_df.columns if c.endswith("_char_len") or c.endswith("_word_len")]
].describe()

length_summary

,objective_text_char_len,objective_text_word_len,competences_knowledge_text_char_len,competences_knowledge_text_word_len,competences_practice_text_char_len,competences_practice_text_word_len,competences_applying_technology_text_char_len,competences_applying_technology_text_word_len,competences_multidisciplinary_text_char_len,competences_multidisciplinary_text_word_len,competences_communication_text_char_len,competences_communication_text_word_len,competences_investigations_text_char_len,competences_investigations_text_word_len,learning_outcomes_text_char_len,learning_outcomes_text_word_len,content_text_char_len,content_text_word_len,prerequisites_text_char_len,prerequisites_text_word_len,embedding_text_main_char_len,embedding_text_main_word_len,embedding_text_enriched_char_len,embedding_text_enriched_word_len,review_text_full_char_len,review_text_full_word_len
count,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000
mean,418.240000,62.780000,149.20000,21.860000,157.720000,24.340000,1.900000,0.30000,19.160000,2.920000,16.840000,2.700000,4.580000,0.720000,22.020000,3.040000,799.140000,115.760000,40.960000,5.140000,1553.580000,227.780000,1691.220000,245.420000,1783.62000,255.860000
std,233.347036,33.727171,83.59792,11.705589,108.904197,17.569233,13.435029,2.12132,68.835017,10.514984,55.570275,8.730079,32.385491,5.091169,101.756109,13.723152,165.254659,24.104839,30.723892,3.350358,401.389479,60.086771,491.894766,71.476581,487.17511,71.059909
min,185.000000,29.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,458.000000,71.000000,0.000000,0.000000,960.000000,146.000000,960.000000,147.000000,1066.00000,159.000000
25%,264.750000,40.000000,82.75000,13.000000,85.250000,13.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,687.500000,99.000000,21.000000,3.000000,1282.500000,188.250000,1344.750000,197.000000,1430.75000,207.250000
50%,306.000000,47.500000,146.50000,22.000000,115.500000,17.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,784.500000,111.500000,39.500000,5.000000,1468.000000,213.500000,1530.500000,224.000000,1628.00000,235.000000
75%,536.500000,77.500000,208.75000,29.750000,227.000000,36.750000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,874.250000,128.250000,52.250000,7.000000,1748.500000,248.750000,1893.250000,271.000000,1991.25000,282.750000
max,1050.000000,147.000000,358.00000,47.000000,455.000000,75.000000,95.000000,15.00000,287.000000,45.000000,284.000000,43.000000,229.000000,36.000000,639.000000,83.000000,1411.000000,187.000000,157.000000,16.000000,2748.000000,384.000000,3113.000000,430.000000,3202.00000,439.000000


### Note on text length and downstream model context

The text-length diagnostics in this notebook are important for interpreting later benchmark results.

In this corpus, the primary course representations are long enough that short-context embedding models may truncate important parts of the text. This matters especially when comparing models with very different context limits.

As a practical interpretation:
- short-context models may lose part of the course description
- medium-context models are likely to handle most courses adequately
- long-context models can process the full representation without truncation concerns

This issue is discussed and quantified in Notebook 03 when model performance is compared against the final text variants.

The newline structure in the embedding text is also intentional. It preserves soft section boundaries between course name, objectives, competences, content, learning outcomes, and prerequisites. Downstream notebooks should avoid collapsing these line breaks unless there is a clear methodological reason to do so.

In [142]:
# Cell 23 - Show average contribution by field

field_contribution_rows = []

for col in [
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
]:
    field_contribution_rows.append({
        "field": col,
        "mean_char_len": round(df[col].apply(char_len).mean(), 2),
        "median_char_len": round(df[col].apply(char_len).median(), 2),
        "mean_word_len": round(df[col].apply(word_len).mean(), 2),
        "median_word_len": round(df[col].apply(word_len).median(), 2),
        "non_empty_rows": int((df[col].str.strip() != "").sum()),
    })

field_contribution_df = pd.DataFrame(field_contribution_rows).sort_values(
    by="mean_char_len",
    ascending=False
).reset_index(drop=True)

field_contribution_df

,field,mean_char_len,median_char_len,mean_word_len,median_word_len,non_empty_rows
0,content_text,799.14,784.5,115.76,111.5,50
1,objective_text,418.24,306.0,62.78,47.5,50
2,competences_practice_text,157.72,115.5,24.34,17.0,45
3,competences_knowledge_text,149.20,146.5,21.86,22.0,47
4,prerequisites_text,40.96,39.5,5.14,5.0,45
5,learning_outcomes_text,22.02,0.0,3.04,0.0,3
6,competences_multidisciplinary_text,19.16,0.0,2.92,0.0,4
7,competences_communication_text,16.84,0.0,2.70,0.0,5
8,competences_investigations_text,4.58,0.0,0.72,0.0,1
9,competences_applying_technology_text,1.90,0.0,0.30,0.0,1


In [143]:
# Cell 24 - Measure how much the enriched representation expands the main representation

enriched_impact_df = df[[
    "course_unit_code",
    "course_name",
    "learning_outcomes_text",
    "prerequisites_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "embedding_text_main",
    "embedding_text_enriched",
]].copy()

enriched_impact_df["main_char_len"] = enriched_impact_df["embedding_text_main"].apply(char_len)
enriched_impact_df["enriched_char_len"] = enriched_impact_df["embedding_text_enriched"].apply(char_len)
enriched_impact_df["added_chars_from_enrichment"] = (
    enriched_impact_df["enriched_char_len"] - enriched_impact_df["main_char_len"]
)

enriched_impact_df["has_learning_outcomes"] = enriched_impact_df["learning_outcomes_text"].str.strip() != ""
enriched_impact_df["has_prerequisites"] = enriched_impact_df["prerequisites_text"].str.strip() != ""
enriched_impact_df["has_applying_technology"] = enriched_impact_df["competences_applying_technology_text"].str.strip() != ""
enriched_impact_df["has_multidisciplinary"] = enriched_impact_df["competences_multidisciplinary_text"].str.strip() != ""
enriched_impact_df["has_communication"] = enriched_impact_df["competences_communication_text"].str.strip() != ""
enriched_impact_df["has_investigations"] = enriched_impact_df["competences_investigations_text"].str.strip() != ""

enriched_impact_df["num_enriched_signal_sources"] = (
    enriched_impact_df[
        [
            "has_learning_outcomes",
            "has_prerequisites",
            "has_applying_technology",
            "has_multidisciplinary",
            "has_communication",
            "has_investigations",
        ]
    ]
    .sum(axis=1)
)

enriched_impact_df[[
    "course_unit_code",
    "course_name",
    "num_enriched_signal_sources",
    "has_learning_outcomes",
    "has_prerequisites",
    "has_applying_technology",
    "has_multidisciplinary",
    "has_communication",
    "has_investigations",
    "added_chars_from_enrichment"
]].sort_values(
    by=["added_chars_from_enrichment", "num_enriched_signal_sources"],
    ascending=[False, False]
).head(15)

,course_unit_code,course_name,num_enriched_signal_sources,has_learning_outcomes,has_prerequisites,has_applying_technology,has_multidisciplinary,has_communication,has_investigations,added_chars_from_enrichment
28,TT00CD97,Concurrent Programming,2,True,True,False,False,False,False,732
40,TT00CE09,Auditing and Penetration Testing,3,False,True,False,True,False,True,662
20,TT00CD90,Software Design,3,False,True,False,True,True,False,469
27,TT00CD96,Graphics Programming,2,True,True,False,False,False,False,456
43,TT00CE12,Security Operations Center,2,False,True,False,False,True,False,395
19,TT00CD89,Software Testing,2,False,True,False,True,False,False,394
38,TT00CE07,Cyber Security Management,2,False,True,False,True,False,False,365
25,TT00CD92,Mobile Project,2,False,True,False,False,True,False,362
26,TT00CD95,Functional Programming,2,True,True,False,False,False,False,245
44,TT00CD88,Hardening,2,False,True,True,False,False,False,205


In [144]:
# Cell 24B - Summarize which enriched-only signal sources activate across courses

enriched_signal_summary = pd.DataFrame({
    "signal_source": [
        "learning_outcomes",
        "prerequisites",
        "applying_technology_to_practice",
        "multidisciplinary_competence",
        "communication_and_teamwork",
        "investigations_and_information_retrieval",
    ],
    "non_empty_rows": [
        int((df["learning_outcomes_text"].str.strip() != "").sum()),
        int((df["prerequisites_text"].str.strip() != "").sum()),
        int((df["competences_applying_technology_text"].str.strip() != "").sum()),
        int((df["competences_multidisciplinary_text"].str.strip() != "").sum()),
        int((df["competences_communication_text"].str.strip() != "").sum()),
        int((df["competences_investigations_text"].str.strip() != "").sum()),
    ]
})

enriched_signal_summary["coverage_ratio"] = (
    enriched_signal_summary["non_empty_rows"] / len(df)
).round(4)

enriched_signal_summary.sort_values(
    by=["non_empty_rows", "signal_source"],
    ascending=[False, True]
).reset_index(drop=True)

,signal_source,non_empty_rows,coverage_ratio
0,prerequisites,45,0.90
1,communication_and_teamwork,5,0.10
2,multidisciplinary_competence,4,0.08
3,learning_outcomes,3,0.06
4,applying_technology_to_practice,1,0.02
5,investigations_and_information_retrieval,1,0.02


In [145]:
# Cell 25 - Check the courses with the longest representations

length_audit_df[[
    "course_unit_code",
    "course_name",
    "embedding_text_main_char_len",
    "embedding_text_enriched_char_len",
]].sort_values(
    by="embedding_text_enriched_char_len",
    ascending=False
).head(10)

,course_unit_code,course_name,embedding_text_main_char_len,embedding_text_enriched_char_len
38,TT00CE07,Cyber Security Management,2748,3113
20,TT00CD90,Software Design,2337,2806
43,TT00CE12,Security Operations Center,2326,2721
19,TT00CD89,Software Testing,2194,2588
12,TT00CD82,Project Management and Quality Assurance,2474,2534
27,TT00CD96,Graphics Programming,1943,2399
40,TT00CE09,Auditing and Penetration Testing,1676,2338
2,TT00CD72,Servers and containers,2252,2314
28,TT00CD97,Concurrent Programming,1519,2251
0,TT00CD70,Data Networks,2108,2144


### 9. Repetitiveness audit for administrative boilerplate

One risk in curriculum-overlap work is adding text that is highly repetitive across many courses, because that can inflate similarity for the wrong reason.

Assessment-related text is preserved in the cleaned dataset for reference, but it is not included in the embedding variants.

This section performs a lightweight audit of repeated phrases in selected fields.

In [146]:
# Cell 26 - Lightweight repeated-text audit

def get_repeated_exact_values(series, min_count=2):
    cleaned_values = [clean_text_value(v) for v in series if clean_text_value(v)]
    counts = Counter(cleaned_values)
    repeated = [(text, count) for text, count in counts.items() if count >= min_count]
    repeated = sorted(repeated, key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(repeated, columns=["text", "count"])

print("Repeated exact values in assessment_pass_fail_criteria:")
repeated_pass_fail_df = get_repeated_exact_values(df["assessment_pass_fail_criteria"], min_count=2)
repeated_pass_fail_df.head(10)

Repeated exact values in assessment_pass_fail_criteria:


,text,count


In [147]:
# Cell 27 - Lightweight repeated-fragment audit for selected descriptive fields

def split_into_candidate_lines(text):
    cleaned = clean_text_value(text)
    if not cleaned:
        return []
    parts = re.split(r"(?<=[\.\;\:])\s+|\n+", cleaned)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

def repeated_fragments_from_series(series, min_count=3, min_chars=30):
    fragments = []
    for value in series:
        fragments.extend([
            frag for frag in split_into_candidate_lines(value)
            if len(frag) >= min_chars
        ])
    counts = Counter(fragments)
    repeated = [(frag, count) for frag, count in counts.items() if count >= min_count]
    repeated = sorted(repeated, key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(repeated, columns=["fragment", "count"])

print("Repeated fragments in objective_text:")
repeated_objective_fragments_df = repeated_fragments_from_series(df["objective_text"], min_count=3, min_chars=40)
repeated_objective_fragments_df.head(15)

Repeated fragments in objective_text:


,fragment,count


### 10. Sample inspection of the final course representation

A preprocessing notebook should include human-readable examples.

This section prints selected courses so that the generated representation can be reviewed directly instead of treated as a black box.

In [148]:
# Cell 28 - Inspect representative course records across different representation patterns

representative_codes = [
    "TT00CD70",  # Data Networks - common fields + prerequisites
    "TT00CD71",  # Linux Basics - common fields without prerequisites
    "TT00CD96",  # Graphics Programming - learning outcomes
    "TT00CD90",  # Software Design - multidisciplinary + communication
    "TT00CD88",  # Hardening - applying technology to practice
    "TT00CE09",  # Auditing and Penetration Testing - investigations + multidisciplinary
]

inspection_df = df[df["course_unit_code"].isin(representative_codes)].copy()

for _, row in inspection_df.iterrows():
    print("=" * 120)
    print(f"Course: {row['course_unit_code']} | {row['course_name']}")
    print("-" * 120)
    print("embedding_text_main")
    print(row["embedding_text_main"])
    print()
    print("embedding_text_enriched")
    print(row["embedding_text_enriched"])
    print()

Course: TT00CD70 | Data Networks
------------------------------------------------------------------------------------------------------------------------
embedding_text_main
Data Networks
You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are familiar with protocols and data transmission principles used on the Internet. You know the layers of the OSI model and identify the technologies and protocols used on each layer. You can design a local area network using switches, routers, workstations, and servers. You are able to design cabling, switching topology, and routing scheme to run the data network. You know the different devices used on a network, such as switches, routers, firewalls, and wireless base stations. You can change, handle, and save the configuration used in network devices. You know how local area networks interconnect to one another through wide area networks.
You know th

In [149]:
# Cell 28B - Compare review text and enriched signals for edge-case courses

edge_case_codes = [
    "TT00CD96",  # Graphics Programming
    "TT00CD97",  # Concurrent Programming
    "TT00CD90",  # Software Design
    "TT00CD88",  # Hardening
    "TT00CE09",  # Auditing and Penetration Testing
]

edge_case_df = df[df["course_unit_code"].isin(edge_case_codes)].copy()

for _, row in edge_case_df.iterrows():
    print("=" * 120)
    print(f"Course: {row['course_unit_code']} | {row['course_name']}")
    print("-" * 120)
    print(row["review_text_full"])
    print()

Course: TT00CD90 | Software Design
------------------------------------------------------------------------------------------------------------------------
Course code: TT00CD90
Course name: Software Design
Objective: After completing the course, you will learn the principles, methods and tools of software engineering. The primary goal is to strengthen the definition and service design skills and to teach how to document the customer's needs in such a way that they are useful for the product development team. You will learn to use the commonly used UML (Unified Modeling Language) description language. The UML description method is introduced to the most common uses, such as use case (Use Case), sequence diagram (Sequence Diagram), state machine description (State Chart) or deployment diagram (Deployment Diagram). To help with the definition, the use of so-called MockUp tool is learned.
Knowledge and understanding: The student knows how to define software solutions that meet the require

### 11. Assemble the final cleaned dataframe

The saved output should support:
- later notebooks
- manual labeling
- benchmark experiments
- the final proof-of-concept UI

For that reason, the saved dataset includes:
- cleaned descriptive fields
- normalized helper fields
- embedding text variants
- review text fields
- reference metadata

In [150]:
# Cell 29 - Select final columns for the cleaned dataset

# assessment_pass_fail_criteria is kept for reference despite very low coverage.
# It is not used in embedding text construction.

final_columns = [
    "degree_programme",
    "course_name",
    "course_unit_code",
    "credits",
    "teaching_language_text",
    "responsible_person_text",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "assessment_grading_scale",
    "assessment_pass_fail_criteria",
    "course_name_normalized",
    "prerequisites_text_normalized",
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_compact",
    "review_text_full",
]

clean_df = df[final_columns].copy()

print("Clean dataframe shape:", clean_df.shape)
clean_df.head(3)

Clean dataframe shape: (50, 24)


,degree_programme,course_name,course_unit_code,credits,teaching_language_text,responsible_person_text,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text,assessment_grading_scale,assessment_pass_fail_criteria,course_name_normalized,prerequisites_text_normalized,embedding_text_main,embedding_text_enriched,review_text_compact,review_text_full
0,Information and Communication Technology,Data Networks,TT00CD70,5,"Finnish, English",Pasi Hyytiäinen,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics,1-5,The student has sufficient knowledge of the theory on network protocols and is able to recognize some sections of existing solutions. The student can design...,data networks,linux basics,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...
1,Information and Communication Technology,Linux Basics,TT00CD71,4,"Finnish, English","Juho Pekki, Teemu Siikaniemi","After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",,1-5,,linux basics,,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera...","Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera..."
2,Information and Communication Technology,Servers and containers,TT00CD72,5,"Finnish, English","Juho Pekki, Teemu Siikaniemi",You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able ...,You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pri...,"You are able to utilize basic cloud services and create virtual machines. You a

### 12. Final diagnostics before saving

These checks confirm that the cleaned output is complete enough for downstream use.

The focus is on:
- empty critical fields
- duplicate identifiers
- representative text lengths
- overall readiness for Notebook 02 and Notebook 03

In [151]:
# Cell 30 - Check critical emptiness after final assembly

critical_empty_summary = {
    "empty_course_name": (clean_df["course_name"].fillna("").str.strip() == "").sum(),
    "empty_course_unit_code": (clean_df["course_unit_code"].fillna("").str.strip() == "").sum(),
    "empty_objective_text": (clean_df["objective_text"].fillna("").str.strip() == "").sum(),
    "empty_content_text": (clean_df["content_text"].fillna("").str.strip() == "").sum(),
    "empty_embedding_text_main": (clean_df["embedding_text_main"].fillna("").str.strip() == "").sum(),
    "empty_embedding_text_enriched": (clean_df["embedding_text_enriched"].fillna("").str.strip() == "").sum(),
    "empty_review_text_full": (clean_df["review_text_full"].fillna("").str.strip() == "").sum(),
}

pd.Series(critical_empty_summary)

empty_course_name                0
empty_course_unit_code           0
empty_objective_text             0
empty_content_text               0
empty_embedding_text_main        0
empty_embedding_text_enriched    0
empty_review_text_full           0
dtype: int64

In [152]:
# Cell 31 - Final text length summary for the main representations

final_length_summary = clean_df[[
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_full",
]].copy()

for col in final_length_summary.columns:
    final_length_summary[f"{col}_char_len"] = final_length_summary[col].apply(char_len)
    final_length_summary[f"{col}_word_len"] = final_length_summary[col].apply(word_len)

final_length_summary[
    [c for c in final_length_summary.columns if c.endswith("_char_len") or c.endswith("_word_len")]
].describe()

,embedding_text_main_char_len,embedding_text_main_word_len,embedding_text_enriched_char_len,embedding_text_enriched_word_len,review_text_full_char_len,review_text_full_word_len
count,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000
mean,1553.580000,227.780000,1691.220000,245.420000,1783.62000,255.860000
std,401.389479,60.086771,491.894766,71.476581,487.17511,71.059909
min,960.000000,146.000000,960.000000,147.000000,1066.00000,159.000000
25%,1282.500000,188.250000,1344.750000,197.000000,1430.75000,207.250000
50%,1468.000000,213.500000,1530.500000,224.000000,1628.00000,235.000000
75%,1748.500000,248.750000,1893.250000,271.000000,1991.25000,282.750000
max,2748.000000,384.000000,3113.000000,430.000000,3202.00000,439.000000


In [153]:
# Cell 32 - Save cleaned dataset to CSV

clean_df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8")

print("Saved cleaned dataset to:", CLEAN_CSV_PATH.resolve())
print("Rows, columns:", clean_df.shape)

Saved cleaned dataset to: C:\Users\user\Downloads\curriculum-overlap-poc\data\courses_clean.csv
Rows, columns: (50, 24)


In [154]:
# Cell 33 - Final preview of the most important output columns

clean_df[[
    "course_unit_code",
    "course_name",
    "objective_text",
    "competences_knowledge_text",
    "competences_practice_text",
    "competences_applying_technology_text",
    "competences_multidisciplinary_text",
    "competences_communication_text",
    "competences_investigations_text",
    "learning_outcomes_text",
    "content_text",
    "prerequisites_text",
    "embedding_text_main",
    "embedding_text_enriched",
    "review_text_full",
]].head(5)

,course_unit_code,course_name,objective_text,competences_knowledge_text,competences_practice_text,competences_applying_technology_text,competences_multidisciplinary_text,competences_communication_text,competences_investigations_text,learning_outcomes_text,content_text,prerequisites_text,embedding_text_main,embedding_text_enriched,review_text_full
0,TT00CD70,Data Networks,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...,You can apply data network skills to communication between devices.,,,,,,"In this course, you will learn the principles of network structure and protocol design, as well as the principles of Internet communication. You will master...",Linux Basics,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...
1,TT00CD71,Linux Basics,"After completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work with and manage t...",You understand the key concepts and architecture of the Linux operating system. You understand the importance of managing user accounts and groups.,You know how to manage the key functions of the Linux system and use command-line tools. You know how to manage the filesystem and user account permissions....,,,,,,"In this course, you will learn how to effectively manage the Linux operating system using a text-based interface. You will understand key concepts and be ab...",,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...","Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera..."
2,TT00CD72,Servers and containers,You understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud computing. You are able ...,You understand the fundamental types and functions of servers. You are able to distinguish between various cloud service models. You understand the core pri...,"You are able to utilize basic cloud services and create virtual machines. You are able to build, run, and manage containers, and use orchestration tools to ...",,,,,,"In this course, you will learn the basic concepts of servers and cloud computing, and differentiate between different cloud service models. You will master ...","Linux Basics, Data Networks, Databases",Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...,Course code: TT00CD72\nCourse name: Servers and containers\nObjective: You understand the fundamental types and functions of servers. You understand the key...
3,TT00CD73,Digital Technology,You understand the basics of digital number systems and know how to apply the knowledge in practice. You can rationalize simple technical problems logically...,You know the concepts and principles

## Final note

This notebook performs conservative and thesis-defensible preprocessing for curriculum-overlap detection.

### What this notebook does
- preserves the real course dataset structure
- flattens nested JSON into reusable tabular fields
- normalizes whitespace and simple formatting noise
- creates two embedding-ready text variants
- prepares fair review text for later human labeling
- audits field coverage, text length, enrichment impact, and repetitiveness
- saves a cleaned dataset for downstream notebooks and the PoC UI

### What this notebook intentionally does not do
- remove stop words
- aggressively rewrite academic language
- inject prerequisite course descriptions into the main text
- include grading criteria in embedding representations
- add administrative metadata such as degree programme or course code into embedding text
- apply handcrafted weighting tricks at preprocessing time

### Why this design is appropriate
The objective of the thesis is not to force a specific model to win.  
The objective is to represent each course as faithfully as possible for the task of curriculum overlap detection.

The selected representation therefore emphasizes:
- course objectives
- common competence statements
- course content
- additional curriculum signals only when they exist and are potentially meaningful

The main representation, `embedding_text_main`, is intentionally conservative.  
It provides a stable and comparable benchmark representation across the full dataset.

The enriched representation, `embedding_text_enriched`, preserves sparse but semantically meaningful curriculum signals such as:
- learning outcomes
- teamwork-related competence statements
- multidisciplinary competence statements
- investigations/information retrieval statements
- prerequisite relations

These enriched signals are not assumed to improve results automatically.  
Instead, they are retained as a controlled comparison so that later notebooks can test whether broader curriculum context improves overlap detection.

In particular, the embedding variants are kept largely **label-free** in order to reduce **structural bias** caused by repeated formatting tokens shared across all course descriptions. At the same time, light semantic cues are retained where relationship meaning matters, especially for prerequisites and learning outcomes. Labeled review text is retained for human readability during manual inspection and annotation.

This keeps the pipeline transparent, reproducible, and suitable for both academic evaluation and practical proof-of-concept use.